# 2D Shallow-Water Weather Dynamics (Full SWE)

This notebook runs a **full shallow-water equation (SWE)** simulator with state channels `[h, u, v]`.
It is designed as a Cartesian, beta-plane proxy for PDEArena-style weather trajectory generation.

## Governing PDE

We evolve fluid depth $h(x,y,t)$ and horizontal velocity $(u,v)$:

$$
\partial_t h + \nabla\cdot(h\mathbf{u}) = 0
$$

$$
\partial_t u + u\partial_x u + v\partial_y u = f(y)\,v - g\,\partial_x h + \nu\nabla^2 u - r\,u
$$

$$
\partial_t v + u\partial_x v + v\partial_y v = -f(y)\,u - g\,\partial_y h + \nu\nabla^2 v - r\,v
$$

with beta-plane Coriolis

$$
f(y)=f_0+\beta(y-y_0).
$$

### Terms used here

- **Advection**: nonlinear transport $u\partial_x(\cdot)+v\partial_y(\cdot)$
- **Pressure-gradient force**: $-g\nabla h$
- **Coriolis coupling**: $(+f v, -f u)$
- **Laplacian viscosity**: $\nu\nabla^2(u,v)$
- **Linear drag (Rayleigh)**: $-r(u,v)$
- **Hyperviscosity filter**: spectral integrating-factor damping at high wavenumber for numerical stability

## Numerical setup

- Spatial derivatives use **FFT pseudo-spectral** operators.
- Time stepping uses **adaptive RK4** with CFL control.
- The domain is a **periodic rectangle** in both directions (implied by FFT).
- Output is `[h, u, v]` at saved times.

## Initial conditions

ICs are built in vorticity space, then mapped to balanced $(h,u,v)$:

1. Random large-scale streamfunction background ($k^{-2}$ spectrum)
2. **Per-column i.i.d. random zonal jet** (PDEArena `:random2` style)
3. Wave-6 Gaussian perturbation in latitude
4. Solve $\nabla^2\psi=\zeta$, then set
   $u=-\partial_y\psi$, $v=\partial_x\psi$, $h=H+(f_0/g)\psi$

This gives geostrophically balanced ICs and avoids spurious startup gravity-wave artifacts.

## Connection to PDEArena

This setup is intentionally close to PDEArena weather generation in spirit:

- Full SWE dynamics (prognostic mass + momentum)
- Beta-plane rotating flow
- Random2-style jet perturbations and wave forcing
- Spectral numerics supporting coherent vortices

Main difference: **geometry**. PDEArena production data is generated on the sphere (SpeedyWeather/Julia), while this notebook uses a periodic Cartesian beta-plane box.

## Why intersting/useful

- Produces rich vortex-dominated trajectories with physically interpretable channels.
- Closer to weather-like dynamics than simple advection-diffusion benchmarks.
- Useful for ML training/evaluation on multi-field, nonlinear, coupled PDE dynamics while remaining fast and controllable.


In [ ]:
import torch
from IPython.display import HTML

from autosim.experimental.simulations import ShallowWater2D
from autosim.utils import plot_spatiotemporal_video


In [ ]:
# Interactive preview config: keep CFL fixed for stability.
sim = ShallowWater2D(
    return_timeseries=True,
    nx=64,
    ny=64,
    Lx=64.0,
    Ly=64.0,
    T=320.0,
    dt_save=1.0,
    cfl=0.12,
    log_level="warning",
    dtype=torch.float32,
    parameters_range={"amp": (0.05, 0.2)},
)

batch = sim.forward_samples_spatiotemporal(n=1)
params = batch["constant_scalars"]
outputs = batch["data"]


In [ ]:
print("constant_scalars shape:", params.shape)
print("data shape:", outputs.shape)
print("sample params (trajectory 0):", {"amp": float(params[0, 0])})


In [ ]:
anim = plot_spatiotemporal_video(
    outputs[:, :100],
    batch_idx=0,
    channel_names=sim.output_names,
)

HTML(anim.to_jshtml())

### Generate broad dataset for ML training


In [ ]:
from autosim.utils import generate_output_data

sim_train = ShallowWater2D(
    return_timeseries=True,
    log_level="warning",
    nx=64,
    ny=128,
    Lx=64.0,
    Ly=128.0,
    T=10.0,
    dt_save=1.0,
    cfl=0.12,
    parameters_range={"amp": (0.05, 0.2)},
)

outputs_data = generate_output_data(sim_train)


In [ ]:
import os

import torch

dataset_name = "swe"
for split in ["train", "valid", "test"]:
    os.makedirs(f"../../../autocast/datasets/{dataset_name}/{split}/", exist_ok=True)
    torch.save(
        outputs_data[split],
        (f"../../../autocast/datasets/{dataset_name}/{split}/data.pt"),
    )
